In [11]:
import threading
import multiprocessing as mp
import time
import os

In [2]:
%%time


N = 10 ** 9
N_THREADS = 10
STEP = N // N_THREADS


def counter(a, b):
    print(f"--- started {threading.current_thread().name=}, {a=}, {b=}")
    while a < b:
        a += 1

    print(f"--- finished {threading.current_thread().name=}")


def run():
    threads = [
        threading.Thread(
            target=counter,
            args=(i * STEP, (i + 1) * STEP),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]
    
    for th in threads:
        th.start()
    
    for th in threads:
        th.join()


run()

--- started threading.current_thread().name='th_counter_0', a=0, b=100000000--- started threading.current_thread().name='th_counter_1', a=100000000, b=200000000

--- started threading.current_thread().name='th_counter_2', a=200000000, b=300000000
--- started threading.current_thread().name='th_counter_3', a=300000000, b=400000000
--- started threading.current_thread().name='th_counter_4', a=400000000, b=500000000
--- started threading.current_thread().name='th_counter_5', a=500000000, b=600000000
--- started threading.current_thread().name='th_counter_6', a=600000000, b=700000000
--- started threading.current_thread().name='th_counter_7', a=700000000, b=800000000
--- started threading.current_thread().name='th_counter_8', a=800000000, b=900000000
--- started threading.current_thread().name='th_counter_9', a=900000000, b=1000000000
--- finished threading.current_thread().name='th_counter_0'--- finished threading.current_thread().name='th_counter_3'

--- finished threading.current_thread

In [3]:
mp.cpu_count()

12

In [4]:
mp.active_children()

[]

In [6]:
mp.get_start_method()

'spawn'

In [7]:
mp.get_all_start_methods()

['spawn', 'fork', 'forkserver']

In [9]:
pr = mp.current_process()

In [10]:
pr.name, pr.daemon, pr.is_alive(), pr.pid

('MainProcess', False, True, 51301)

In [13]:
ctx = mp.get_context("fork")

In [15]:
%%time


N = 10 ** 9
N_PROCS = 10
STEP = N // N_PROCS


def counter(a, b, lock):
    with lock:
        print(f"--- started {os.getpid()=}, {a=}, {b=}")

    while a < b:
        a += 1

    with lock:
        print(f"--- finished {os.getpid()=}")


def run():
    lock = ctx.Lock()
    procs = [
        ctx.Process(
            target=counter,
            args=(i * STEP, (i + 1) * STEP, lock),
            name=f"pr_counter_{i}",
        )
        for i in range(N_PROCS)
    ]
    
    for pr in procs:
        pr.start()
    
    for pr in procs:
        pr.join()


run()

--- started os.getpid()=97651, a=0, b=100000000
--- started os.getpid()=97653, a=100000000, b=200000000
--- started os.getpid()=97655, a=200000000, b=300000000
--- started os.getpid()=97656, a=300000000, b=400000000
--- started os.getpid()=97657, a=400000000, b=500000000
--- started os.getpid()=97658, a=500000000, b=600000000
--- started os.getpid()=97659, a=600000000, b=700000000
--- started os.getpid()=97660, a=700000000, b=800000000
--- started os.getpid()=97661, a=800000000, b=900000000
--- started os.getpid()=97662, a=900000000, b=1000000000
--- finished os.getpid()=97651
--- finished os.getpid()=97657
--- finished os.getpid()=97653
--- finished os.getpid()=97655
--- finished os.getpid()=97661
--- finished os.getpid()=97660
--- finished os.getpid()=97658
--- finished os.getpid()=97662
--- finished os.getpid()=97656
--- finished os.getpid()=97659
CPU times: user 21.1 ms, sys: 52.1 ms, total: 73.2 ms
Wall time: 1.56 s


In [20]:
import queue
import requests

In [21]:
%%time


N = 20
N_THREADS = 5
STEP = N // N_THREADS

URL = "https://www.python.org/"
URLS = [URL] * N


def fetch_urls(que):
    print(f"--- started {threading.current_thread().name=}")
    t1 = time.time()
    cnt = 0
    
    while True:
        url = que.get()
        if url is None:
            que.put(url)
            break

        cnt += 1
        resp = requests.get(url)
        #time.sleep(1)
        print(resp.status_code)

    t2 = time.time()
    print(f"--- finished {threading.current_thread().name=}", t2 - t1, cnt)


def run():
    que = queue.Queue()
    for url in URLS:
        que.put(url)
    que.put(None)

    threads = [
        threading.Thread(
            target=fetch_urls,
            args=(que,),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]
    
    for th in threads:
        th.start()
    
    for th in threads:
        th.join()


run()

--- started threading.current_thread().name='th_counter_0'--- started threading.current_thread().name='th_counter_1'

--- started threading.current_thread().name='th_counter_2'
--- started threading.current_thread().name='th_counter_3'
--- started threading.current_thread().name='th_counter_4'
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
--- finished threading.current_thread().name='th_counter_4' 6.178775072097778 4
200
--- finished threading.current_thread().name='th_counter_3' 6.2923359870910645 4
200
--- finished threading.current_thread().name='th_counter_1' 6.295859098434448 3
200
--- finished threading.current_thread().name='th_counter_2' 6.322051763534546 4
200
--- finished threading.current_thread().name='th_counter_0' 6.4728100299835205 5
CPU times: user 2.23 s, sys: 79.6 ms, total: 2.31 s
Wall time: 6.48 s


In [23]:
%%time


N = 20
N_THREADS = 5
STEP = N // N_THREADS

URL = "https://www.python.org/"
URLS = [URL] * N


def fetch_urls(que, sem):
    print(f"--- started {threading.current_thread().name=}")
    t1 = time.time()
    cnt = 0
    
    while True:
        url = que.get()
        if url is None:
            que.put(url)
            break

        cnt += 1
        with sem:
            resp = requests.get(url)
        #time.sleep(1)
        print(resp.status_code)

    t2 = time.time()
    print(f"--- finished {threading.current_thread().name=}", t2 - t1, cnt)


def run():
    que = queue.Queue()
    for url in URLS:
        que.put(url)
    que.put(None)

    sem = threading.Semaphore(2)

    threads = [
        threading.Thread(
            target=fetch_urls,
            args=(que, sem),
            name=f"th_counter_{i}",
        )
        for i in range(N_THREADS)
    ]
    
    for th in threads:
        th.start()
    
    for th in threads:
        th.join()


run()

--- started threading.current_thread().name='th_counter_0'
--- started threading.current_thread().name='th_counter_1'
--- started threading.current_thread().name='th_counter_2'
--- started threading.current_thread().name='th_counter_3'
--- started threading.current_thread().name='th_counter_4'
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
--- finished threading.current_thread().name='th_counter_2' 5.147075891494751 7
200
--- finished threading.current_thread().name='th_counter_1' 5.179937124252319 3
200
--- finished threading.current_thread().name='th_counter_4' 5.349423170089722 3
200
--- finished threading.current_thread().name='th_counter_0' 5.4067299365997314 4
200
--- finished threading.current_thread().name='th_counter_3' 5.536545038223267 3
CPU times: user 678 ms, sys: 62.5 ms, total: 740 ms
Wall time: 5.54 s


In [27]:
class User:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"User[{self.name=}, id={id(self)}]"


def update_data(data):
    print(f"update_data start data={data}")
    data["to_update"] = "updated"
    data[("1", 2, (3, "4"))] = "1234"
    data["user"] = User("steve")

    print(f"update_data finish data={data}")


data = {"to_update": "init"}

In [28]:
update_data(data)

update_data start data={'to_update': 'init'}
update_data finish data={'to_update': 'updated', ('1', 2, (3, '4')): '1234', 'user': User[self.name='steve', id=4507845024]}


In [29]:
data

{'to_update': 'updated',
 ('1', 2, (3, '4')): '1234',
 'user': User[self.name='steve', id=4507845024]}

In [30]:
data = {"to_update": "init"}

pr = ctx.Process(target=update_data, args=(data,))
pr.start()
pr.join()


print("data after process", data)

update_data start data={'to_update': 'init'}
update_data finish data={'to_update': 'updated', ('1', 2, (3, '4')): '1234', 'user': User[self.name='steve', id=4496262480]}
data after process {'to_update': 'init'}


In [37]:
data = {"to_update": "init"}


with ctx.Manager() as manager:
    manager_data = manager.dict(data)

    pr = ctx.Process(target=update_data, args=(manager_data,))
    pr.start()
    pr.join()

    print("mananger_data after", manager_data)

    data = dict(manager_data)


print("data after process", data)

update_data start data={'to_update': 'init'}
update_data finish data={'to_update': 'updated', ('1', 2, (3, '4')): '1234', 'user': User[self.name='steve', id=4509257968]}
mananger_data after {'to_update': 'updated', ('1', 2, (3, '4')): '1234', 'user': User[self.name='steve', id=4509257968]}
data after process {'to_update': 'updated', ('1', 2, (3, '4')): '1234', 'user': User[self.name='steve', id=4509257664]}


In [38]:
import socket

In [43]:
def client(data):
    print("start_client", data)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect(("localhost", 12_001))
        sock.sendall(data.encode())

        data = sock.recv(1024).decode()
        print(f"client got {data=}")
        
    print("client finished", data)


def server():
    print("start_server")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEPORT, 1)
        sock.bind(("localhost", 12_001))
        sock.listen(5)

        while True:
            client_sock, addr = sock.accept()
            print("server connected from", addr)

            data = client_sock.recv(1024).decode()
            print(f"server got {data=}")
    
            client_sock.sendall(data.upper().encode())

    print("server finished")


pr_server = ctx.Process(target=server)
pr_client = ctx.Process(target=client, args=("данные",))

pr_server.start()
pr_client.start()

start_server
start_client данные
server connected from ('127.0.0.1', 53869)
server got data='данные'
client got data='ДАННЫЕ'
client finished ДАННЫЕ
server connected from ('127.0.0.1', 53896)
server got data='другие данные'


In [44]:
client("другие данные")

start_client другие данные
client got data='ДРУГИЕ ДАННЫЕ'
client finished ДРУГИЕ ДАННЫЕ
